In [ ]:
# notebooks/02_explore_data.ipynb  セル1
%load_ext autoreload
%autoreload 2

import sys

import pandas as pd
from statsbombpy import sb

# src フォルダを import の探索先に追加する(後で自作の loader.py を使うため)
sys.path.append("../src")

# 表の表示設定(列が省略されにくくなる)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)


In [ ]:
# notebooks/02_explore_data.ipynb  セル2
toy = pd.DataFrame(
    {
        "team": ["A", "B", "A", "B"],
        "player": ["p1", "p2", "p3", "p4"],
        "shots": [3, 5, 2, 1],
    }
)
toy


In [ ]:
# notebooks/02_explore_data.ipynb  セル3
print(toy["shots"])                        # 1列を取り出す
print(toy[toy["team"] == "A"])             # 条件に合う行だけを取り出す
print(toy.groupby("team")["shots"].sum())  # チームごとに合計する


In [ ]:
# notebooks/02_explore_data.ipynb  セル4
competitions = sb.competitions()
print(competitions.shape)
competitions.head()


In [ ]:
# notebooks/02_explore_data.ipynb  セル5
print(list(competitions.columns))


In [ ]:
# notebooks/02_explore_data.ipynb  セル6
wc = competitions[competitions["competition_name"].str.contains("World Cup", na=False)]
wc[["competition_id", "season_id", "competition_name", "season_name", "competition_gender"]]


In [ ]:
# notebooks/02_explore_data.ipynb  セル7(課題1)
matches_2022 = sb.matches(competition_id=43, season_id=106)
print(matches_2022.shape)


In [ ]:
# notebooks/02_explore_data.ipynb  セル8
print(list(matches_2022.columns))
matches_2022[["match_id", "match_date", "home_team", "away_team", "home_score", "away_score"]].head()


In [ ]:
# notebooks/02_explore_data.ipynb  セル9
frames = []
for _, row in wc.iterrows():
    matches = sb.matches(
        competition_id=row["competition_id"],
        season_id=row["season_id"],
    )
    frames.append(matches)
    print(row["competition_name"], row["season_name"], len(matches))

all_matches = pd.concat(frames, ignore_index=True)
print(all_matches.shape)


In [ ]:
# notebooks/02_explore_data.ipynb  セル10
teams = pd.concat([all_matches["home_team"], all_matches["away_team"]]).unique()
japan_names = sorted([t for t in teams if "Japan" in t])
print(japan_names)


In [ ]:
# notebooks/02_explore_data.ipynb  セル11
is_japan = all_matches["home_team"].isin(japan_names) | all_matches["away_team"].isin(japan_names)
japan_matches = all_matches[is_japan].sort_values("match_date").reset_index(drop=True)

japan_matches[
    ["match_id", "match_date", "competition", "season", "competition_stage",
     "home_team", "away_team", "home_score", "away_score"]
]


In [ ]:
# notebooks/02_explore_data.ipynb  セル13(課題2)
TEAM_NAME = "Japan"   # japan_names の出力から、使うチーム名を一字一句そのままコピーする
MATCH_ID = 7584      # セル11の表から選んだ match_id(数字。引用符 " " は付けない)

match_row = japan_matches[japan_matches["match_id"] == MATCH_ID]
print(match_row[["match_date", "home_team", "away_team", "home_score", "away_score"]])


In [ ]:
# notebooks/02_explore_data.ipynb  セル14
events = sb.events(match_id=MATCH_ID)
print(events.shape)
events.head()


In [ ]:
# notebooks/02_explore_data.ipynb  セル15
print(list(events.columns))
print(events["type"].value_counts().head(10))
print(events["team"].unique())


In [ ]:
# notebooks/02_explore_data.ipynb  セル16(課題3)
passes = events[events["type"] == "Pass"].copy()
shots = events[events["type"] == "Shot"].copy()

passes["x"] = passes["location"].str[0]
passes["y"] = passes["location"].str[1]

print(len(passes), len(shots))
passes[["minute", "team", "player", "location", "x", "y"]].head()


In [ ]:
# notebooks/02_explore_data.ipynb  セル17
print(passes[["x", "y"]].describe())
print(passes["pass_outcome"].value_counts(dropna=False))


In [ ]:
# notebooks/02_explore_data.ipynb  セル18(課題4)
pass_count = passes.groupby("team").size()
print(pass_count)

shot_summary = shots.groupby("team")["shot_statsbomb_xg"].agg(["count", "sum"])
print(shot_summary)


In [ ]:
# notebooks/02_explore_data.ipynb  セル19
from pitchviz.loader import add_xy, filter_events, get_competitions, get_events, get_team_matches

wc_check = get_competitions("World Cup")
japan_check = get_team_matches(wc_check, "Japan")   # 日本代表が見つからなかった場合は、選んだチームの名前の一部に変える
events_check = get_events(MATCH_ID)

passes_check = add_xy(filter_events(events_check, "Pass", TEAM_NAME))
shots_check = filter_events(events_check, "Shot", TEAM_NAME)

print(len(japan_check), len(japan_matches))
print(len(events_check) == len(events))
print(len(passes_check), len(shots_check))
passes_check[["minute", "player", "x", "y"]].head()
